# On the design of schemas for RDAPS correlation data 

## Introduction

The xradio module defines a processing set to store radio astronomical data (both for single dish instruments and interferometers) on top of Python's xarray data structures. Currently the top-level "processing set" is defined as a DataTree of Xarray Data Sets of a specific format, whose schema itself is defined by Python code. There is a need for a parallel data format to store the parameters generated during calibration of the data. The working name for this is a "Correction Set", and it needs a schema too!

We create the calibration set module beside the processing_set:

```
xradio
└─ processing_set
└─ calibration_set
```

And then we get on with the serious business of helping ourself to schema types already defined for `processing_set`.

```
from xradio.measurement_set.schema import (
    CreatorDict,
    AntennaNameArray,
    AntennaName,
    ScanArray,
    Time,
    TimeCoordArray,
    Frequency,
    FrequencyArray,
    ReceptorLabel,
)
```

The most notable thing about the CorrectionXDS's schema is that we need to accommodate a wide variety of calibration tables, each with its own distinctive set of parameters. In the Measurement Set XDS, every coordinate has a known name and the values of the coordinate have a known type. Here we are, out of necessity, operating a level higher: our we have a coordinate which contains parameter *names* and we will need to have an auxiliary XDS in which the parameter names are mapped to their types. (That auxilliary XDS is not yet defined; it is the most important thing I know to be still missing.)

Expressing the paramater name requirements idiomatically we have:

a type for the names:

```
CalibrationParameterName = Literal["calibration_parameter_name"]
""" Dimension for names of calibration parameters"""
```

a dimension formed of a list of those names:

```
@xarray_dataarray_schema
class CalibrationParameterNameArray:
    """
    Model of the calibration_parameter_name coordinate used in the main dataset.
    """

    data: Data[CalibrationParameterName, str]
    """Name for each parameter."""
```

and two kinds of schemas for the dimensions of containers; one for baseline corrections and one for antenna corrections:

```
@xarray_dataarray_schema
class AntennaCalibrationParameterArray:
    """
    Calibration parameters; these can be real or complex
    """

    data: Data[
        tuple[Time, AntennaName, Frequency, CalibrationParameterName, ReceptorLabel],
        Union[numpy.float32, numpy.float64, numpy.complex64, numpy.complex128],
    ]
```

```
@xarray_dataarray_schema
class BaselineCalibrationParameterArray:
    """
    Calibration parameters for antennas; these can be real or complex
    """

    data: Data[
        tuple[Time, BaselineId, Frequency, CalibrationParameterName, ReceptorLabel],
        Union[numpy.float32, numpy.float64, numpy.complex64, numpy.complex128],
    ]
```

This follows the model of the Measurement Set schema in which Visibility and Spectrum XDSes exist in parallel as distinct top-level objects, with distinct dataarray schemas, but where all their optional multidimensional component arrays (FLAGS, WEIGHTS) use a unified dataarray schema.



Although our FLAG and PARAMETER_ERROR data variables are not optional, I have chosen for now to imitate this organisation for them anyway:

```
@xarray_dataarray_schema
class FlagArray:
    """
    An array of Boolean or integer values with the same shape as the calibration parameters (either baseline or antenna based),
    representing the cumulative flags applying to this data matrix.
    """

    data: Data[
        Union[
            tuple[
                Time, AntennaName, Frequency, CalibrationParameterName, ReceptorLabel
            ],
            tuple[Time, BaselineId, Frequency, CalibrationParameterName, Polarization],
        ],
        bool,
    ]
    """ Flag value.  Data is flagged as bad if the array element is
    ``True`` or nonzero."""
    time: Coordof[TimeCoordArray]
    baseline_id: Optional[Coordof[BaselineArray]]  # Only IF
    antenna_name: Optional[Coordof[AntennaNameArray]]  # Only SD
    frequency: Coordof[FrequencyArray]
    receptor_label: Optional[Coordof[ReceptorLabelArray]] = None
    polarization: Optional[Coordof[PolarizationArray]] = None
    long_name: Optional[Attr[str]] = "Calibration flags"
```

It is required that an antenna-based calibration instantiates the `antenna_name` and `receptor_label` coordinates while a baseline calibration instantiates the `baseline_id` and `polarization` coordinates, but the schema doesn't enforce this. Having those coordinates explicit in the top-level (Antenna|Baseline)CalibrationParamaterArrays mitigates the likely harm of this. At least, I infer this to be the logic behind the corresponding decision in the Measurement Set schema.

I can't say I am 100% thrilled that the schema has this much slack, but I wouldn't really enjoy duplication every sub-schema either, and the schema language doesn't seem to be expressive enough to not choose one of these solutions.

